<a href="https://colab.research.google.com/github/ai1108/kebbi-timebox-love-story/blob/main/kebbi_timebox_text.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Colab 左邊點 🔑 Secrets，新增以下三組密鑰（貼上你自己的值，不要寫在這裡）：

LINE_CHANNEL_ACCESS_TOKEN
LINE_GROUP_ID
NGROK_AUTHTOKEN

In [ ]:
!pip -q install paho-mqtt requests

In [ ]:
import uuid
import time
import requests
import paho.mqtt.client as mqtt

from google.colab import userdata


# ========================================
# LINE 設定
# ========================================

LINE_TOKEN = userdata.get("LINE_CHANNEL_ACCESS_TOKEN")
LINE_GROUP_ID = userdata.get("LINE_GROUP_ID")


# ========================================
# MQTT 設定
# ========================================

MQTT_BROKER = "broker.emqx.io"
MQTT_PORT = 1883

MQTT_TOPIC = "kebbi_timebox_83026"


# ========================================
# 傳送訊息到 LINE
# ========================================

def send_to_line(text):

    url = "https://api.line.me/v2/bot/message/push"

    headers = {
        "Authorization": f"Bearer {LINE_TOKEN}",
        "Content-Type": "application/json"
    }

    data = {
        "to": LINE_GROUP_ID,
        "messages": [
            {
                "type": "text",
                "text": text
            }
        ]
    }

    try:

        response = requests.post(
            url,
            headers=headers,
            json=data,
            timeout=15
        )

        if response.status_code == 200:

            print("✅ LINE 發送成功！")

        else:

            print("❌ LINE 發送失敗")
            print("HTTP =", response.status_code)
            print(response.text)

    except Exception as e:

        print("❌ LINE 發送發生錯誤：")
        print(e)


# ========================================
# MQTT 連線成功時
# ========================================

def on_connect(client, userdata, flags, reason_code, properties):

    print("MQTT 回傳 =", reason_code)

    if reason_code == 0:

        print("✅ MQTT 連線成功")

        client.subscribe(MQTT_TOPIC)

        print("📡 正在等待凱比...")
        print("Topic =", MQTT_TOPIC)

    else:

        print("❌ MQTT 連線失敗")


# ========================================
# 收到凱比資料
# ========================================

def on_message(client, userdata, msg):

    try:

        text = msg.payload.decode("utf-8")

        print("")
        print("==============================")
        print("🤖 收到凱比資料！")
        print("Topic：", msg.topic)
        print("")
        print(text)
        print("==============================")
        print("")

        # 收到凱比資料後直接傳 LINE
        send_to_line(text)

    except Exception as e:

        print("❌ MQTT 訊息處理失敗")
        print(e)


# ========================================
# 建立 MQTT Client
# ========================================

client = mqtt.Client(
    mqtt.CallbackAPIVersion.VERSION2,
    client_id="kebbi_colab_" + uuid.uuid4().hex[:8]
)

client.on_connect = on_connect
client.on_message = on_message


# ========================================
# 連接 MQTT
# ========================================

print("🔄 正在連接 MQTT...")

client.connect(
    MQTT_BROKER,
    MQTT_PORT,
    keepalive=60
)

client.loop_start()

time.sleep(3)

 **沒有凱比時怎麼測**

In [ ]:
test_memory = """【時光寶盒－爺奶戀愛史】
第一次約會：我們第一次約會是在西門町看電影。
看電影趣事：阿嬤看到恐怖片的時候嚇了一大跳，還把爆米花打翻了。"""

info = client.publish(
    "kebbi_timebox_83026",
    test_memory
)

info.wait_for_publish()

print("測試訊息發送結果 =", info.rc)

**真的拿到凱比**

In [ ]:
摸凱比頭
↓
凱比打招呼
↓
問第一次約會
↓
語音辨識
↓
answer_1
↓
問電影趣事
↓
語音辨識
↓
answer_2
↓
組合 memory_text
↓
CodeLab：

發送消息 memory_text
至 Topic kebbi_timebox_83026

↓
EMQX
↓
Colab 自動收到
↓
send_to_line()
↓
LINE 家庭群組